# 한국 우량 우선주 vs 보통주 10년 비교 · 괴리율 백분위 판독기 (Google Colab)

**목적:** 국내 대표 배당 우선주와 본주(보통주)를 **최근 10년 실제 시세·배당 기준**으로 비교하여,
1. **10년 장기 보유 시 보통주 vs 우선주 총수익률(배당 포함)** 대결 결과
2. **현재 가격비율(`우선주 ÷ 보통주`)과 괴리율(`1 − 가격비율`)이 지난 10년 중 상위 몇 %로 싼지/비싼지**
3. **신규 적립 시 지금 가격에서 우선주가 유리한지, 본주가 유리한지**
를 **API 키 없이** 원클릭으로 진단하고 A4 PDF/HTML 리포트를 Google Drive에 저장합니다.

### 자동 개인화 (비공개 Drive 설정 연동)
- Google Drive에 기존 비공개 설정 파일(`MyDrive/Colab Notebooks/dividend_screener/my_kr_portfolio.yaml`)이 있으면 **보유 중인 우선주에 자동으로 「보유중」 배지**를 붙여 최상단에 우선 표시합니다. *(공개 저장소에는 개인 보유 정보가 전혀 포함되지 않습니다.)*
- `PREF_CONFIG["CUSTOM_THRESHOLDS"]`에 본인만의 재검토 기준선(예: 현대차3우B 10년 중앙값 `0.559`)을 넣어두면 차트와 판정표에 기준선 돌파 여부가 자동 표시됩니다.


In [ ]:
# ==============================================================================
# 1. 환경 설정 · 한글 폰트(fonts-nanum) 및 PDF 도구 자동 설치 (API 키 불필요)
# ==============================================================================
import os, sys, re, io, math, time, base64, warnings
from pathlib import Path
from datetime import datetime, timezone, timedelta
from html import escape
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[환경] Google Colab 감지 -> 한글 폰트(fonts-nanum) 및 PDF 엔진 설치 중...")
    subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "libpango-1.0-0", "libpangoft2-1.0-0"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "weasyprint", "pyyaml"], check=False)

import numpy as np
import pandas as pd
import requests
import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mtick

try:
    import yaml
except ImportError:
    yaml = None

try:
    from IPython.display import display
except ImportError:
    display = print

def setup_korean_font():
    chosen = None
    for p in [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothicBold.ttf",
    ]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            if chosen is None:
                chosen = fm.FontProperties(fname=p).get_name()
    if chosen is None:
        avail = {f.name for f in fm.fontManager.ttflist}
        for cand in ["NanumGothic", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]:
            if cand in avail:
                chosen = cand
                break
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        pass
    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 120
    return chosen

FONT_NAME = setup_korean_font()
print(f"[완료] 한글 폰트: {FONT_NAME} | Python {sys.version.split()[0]} | Colab={IN_COLAB}")

In [ ]:
# ==============================================================================
# 2. 추적 대상 우선주 페어 설정 · Google Drive 연결 (비공개 보유종목 자동 태그)
# ==============================================================================
PREF_CONFIG = {
    # 추적할 (우선주 티커, 보통주 티커, 우선주명, 보통주명, 정관 추가배당 원)
    "PAIRS": [
        ("005935", "005930", "삼성전자우",   "삼성전자",     1),
        ("005389", "005380", "현대차3우B",   "현대차",      50),
        ("005387", "005380", "현대차2우B",   "현대차",     100),
        ("005385", "005380", "현대차우",     "현대차",      50),
        ("000815", "000810", "삼성화재우",   "삼성화재",     5),
        ("003555", "003550", "LG우",         "LG",          50),
        ("051915", "051910", "LG화학우",     "LG화학",      50),
        ("051905", "051900", "LG생활건강우", "LG생활건강",  50),
        ("071055", "071050", "한국금융지주우","한국금융지주", 50),
        ("005945", "005940", "NH투자증권우", "NH투자증권",  50),
    ],

    # 선택: 특정 우선주의 개인 재검토 가격비율 기준선 (우선주 ÷ 보통주)
    # 예: 현대차3우B 가격비율이 0.559 위로 올라가면 '할인 축소(재검토선 돌파)' 알림 표시
    "CUSTOM_THRESHOLDS": {
        "005389": {"ratio_high": 0.559, "note": "10년 중앙값 0.559 초과 시 교체 재검토"},
    },

    "LOOKBACK_YEARS": 10,          # 괴리율 분포 및 총수익률 비교 기간 (년)
    "MIN_LIQ_EOK": 5.0,            # 우선주 60일 일평균 거래대금 최소 기준 (억 원)

    # Google Drive 저장 및 비공개 포트폴리오 연동
    "USE_GOOGLE_DRIVE": True,
    "DRIVE_DIR": "/content/drive/MyDrive/Colab Notebooks/dividend_preferred",
    "PORTFOLIO_YAML": "/content/drive/MyDrive/Colab Notebooks/dividend_screener/my_kr_portfolio.yaml",
}

DRIVE_ROOT = None
DRIVE_REPORT_DIR = None
HELD_PREF_SET = set()

if IN_COLAB and PREF_CONFIG["USE_GOOGLE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(PREF_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[Drive 연결 완료] 리포트 저장 경로: {DRIVE_REPORT_DIR}")

        pf_p = Path(PREF_CONFIG["PORTFOLIO_YAML"])
        if yaml is not None and pf_p.exists():
            pf_obj = yaml.safe_load(pf_p.read_text(encoding="utf-8")) or {}
            for tk, info in (pf_obj.get("overrides") or {}).items():
                if isinstance(info, dict) and info.get("common_ticker"):
                    HELD_PREF_SET.add(str(tk).zfill(6))
            if HELD_PREF_SET:
                print(f"[보유 우선주 자동 감지] {sorted(HELD_PREF_SET)} -> 리포트 최상단 강조 표시")
    except Exception as e:
        print(f"[Drive 미연결] 로컬 저장으로 진행합니다 ({type(e).__name__}: {e})")

print(f"[설정 완료] 총 {len(PREF_CONFIG['PAIRS'])}개 우선주-보통주 페어 | 기준 기간 {PREF_CONFIG['LOOKBACK_YEARS']}년")

In [ ]:
# ==============================================================================
# 3. 10년 일별 수정주가(네이버 fchart) + 10년 배당 포함 총수익률(Yahoo Finance) 수집
# ==============================================================================
_FCHART_URL = "https://fchart.stock.naver.com/sise.nhn"
_ITEM_RE = re.compile(r'<item data="([^"]+)"')
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-preferred-tracker)"}

def fetch_naver_daily(ticker: str, count: int = 3000) -> pd.DataFrame:
    for attempt in range(3):
        try:
            r = requests.get(
                _FCHART_URL,
                params={"symbol": ticker, "timeframe": "day", "count": count, "requestType": 0},
                headers=_HEADERS,
                timeout=20,
            )
            r.raise_for_status()
            rows = []
            for m in _ITEM_RE.finditer(r.content.decode("euc-kr", errors="ignore")):
                parts = m.group(1).split("|")
                c, v = float(parts[4]), float(parts[5])
                if c > 0:
                    rows.append((pd.Timestamp(parts[0]), c, v))
            if rows:
                df = pd.DataFrame(rows, columns=["date", "close", "volume"]).set_index("date").sort_index()
                return df[~df.index.duplicated(keep="last")]
        except Exception:
            time.sleep(0.35 * (attempt + 1))
    return pd.DataFrame()

def fetch_yf_tr_and_div(ticker: str) -> tuple[pd.Series, float | None]:
    """Yahoo Finance .KS 10년 수정종가(배당재투자 TR) & 최근 1년 현금배당 합계 조회."""
    try:
        r = requests.get(
            f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}.KS",
            params={"range": "10y", "interval": "1d", "events": "div"},
            headers=_HEADERS,
            timeout=20,
        )
        r.raise_for_status()
        res = r.json()["chart"]["result"][0]
        ts = res["timestamp"]
        adj = res["indicators"].get("adjclose", [{}])[0].get("adjclose")
        vals = adj if adj is not None else res["indicators"]["quote"][0]["close"]
        idx = [pd.Timestamp(datetime.fromtimestamp(t, tz=timezone.utc).strftime("%Y-%m-%d")) for t in ts]
        s = pd.Series(vals, index=idx, dtype=float).dropna()
        s = s[~s.index.duplicated(keep="last")].sort_index()

        # 최근 12개월 현금배당 합계 (Yahoo div events 기준)
        divs = res.get("events", {}).get("dividends", {})
        if divs:
            latest_ts = max(ts)
            cutoff = latest_ts - 375 * 86400
            ttm_div = sum(float(v.get("amount", 0.0)) for k, v in divs.items() if int(k) >= cutoff)
        else:
            ttm_div = None
        return s, (ttm_div if (ttm_div and ttm_div > 0) else None)
    except Exception:
        return pd.Series(dtype=float), None

all_tks = sorted({tk for p in PREF_CONFIG["PAIRS"] for tk in (p[0], p[1])})
print(f"[시세 수집] {len(all_tks)}개 종목 네이버 3,000일 일봉 + Yahoo 10년 TR/배당 수집 중...")
NAVER_MAP = {}
YF_TR_MAP = {}
YF_DIV_MAP = {}

with ThreadPoolExecutor(max_workers=8) as ex:
    f_nav = {ex.submit(fetch_naver_daily, tk): tk for tk in all_tks}
    f_yf = {ex.submit(fetch_yf_tr_and_div, tk): tk for tk in all_tks}
    for fut in as_completed(f_nav):
        NAVER_MAP[f_nav[fut]] = fut.result()
    for fut in as_completed(f_yf):
        s_tr, ttm_d = fut.result()
        YF_TR_MAP[f_yf[fut]] = s_tr
        YF_DIV_MAP[f_yf[fut]] = ttm_d

print(f"[완료] 네이버 일봉 {sum(not d.empty for d in NAVER_MAP.values())}/{len(all_tks)} | Yahoo 10Y TR {sum(len(s)>100 for s in YF_TR_MAP.values())}/{len(all_tks)}")

In [ ]:
# ==============================================================================
# 4. 괴리율 10년 백분위 · 배당수익률 스프레드 · 10년 TR 대결 · 매수 판독
# ==============================================================================
KST = timezone(timedelta(hours=9))
as_of_str = datetime.now(KST).strftime("%Y-%m-%d")
run_stamp = datetime.now(KST).strftime("%Y-%m-%d_%H%M%S")

PAIR_RESULTS = []
RATIO_SERIES = {}

for p_tk, c_tk, p_nm, c_nm, extra_dps in PREF_CONFIG["PAIRS"]:
    df_p, df_c = NAVER_MAP.get(p_tk), NAVER_MAP.get(c_tk)
    if df_p is None or df_c is None or df_p.empty or df_c.empty:
        continue

    common_idx = df_p.index.intersection(df_c.index)
    end_dt = common_idx[-1]
    start_10y = end_dt - pd.Timedelta(days=int(365.25 * PREF_CONFIG["LOOKBACK_YEARS"]))
    sub_idx = common_idx[common_idx >= start_10y]
    if len(sub_idx) < 200:
        continue

    p_cl = df_p.loc[sub_idx, "close"]
    c_cl = df_c.loc[sub_idx, "close"]
    ratio = (p_cl / c_cl).dropna()
    RATIO_SERIES[p_tk] = ratio

    cur_p = float(p_cl.iloc[-1])
    cur_c = float(c_cl.iloc[-1])
    cur_ratio = float(ratio.iloc[-1])
    cur_disc = 1.0 - cur_ratio
    med_ratio = float(ratio.median())
    q25_ratio = float(ratio.quantile(0.25))
    q75_ratio = float(ratio.quantile(0.75))
    # 현재 가격비율의 10년 백분위 (0% = 10년 최저/가장 쌈, 100% = 10년 최고/가장 비쌈)
    pct_pos = float((ratio <= cur_ratio).mean() * 100.0)

    # 최근 60일 일평균 거래대금 (억 원)
    liq_p_eok = float((df_p["close"] * df_p["volume"]).tail(60).mean()) / 1e8
    liq_c_eok = float((df_c["close"] * df_c["volume"]).tail(60).mean()) / 1e8

    # 최근 1년 배당수익률 추정 (보통주 TTM 배당 기준 + 정관 추가배당 반영)
    div_c = YF_DIV_MAP.get(c_tk)
    div_p = YF_DIV_MAP.get(p_tk)
    if div_p is None and div_c is not None:
        div_p = div_c + extra_dps
    yld_c = (div_c / cur_c) if (div_c and cur_c > 0) else None
    yld_p = (div_p / cur_p) if (div_p and cur_p > 0) else None
    yld_spread = (yld_p - yld_c) if (yld_p is not None and yld_c is not None) else None

    # 10년 총수익률(TR, 배당 재투자 포함) 비교 — Yahoo 10Y adjclose 기준
    tr_p_s, tr_c_s = YF_TR_MAP.get(p_tk), YF_TR_MAP.get(c_tk)
    tr_p_cagr = tr_c_cagr = tr_diff = None
    if tr_p_s is not None and tr_c_s is not None and len(tr_p_s) > 250 and len(tr_c_s) > 250:
        t_idx = tr_p_s.index.intersection(tr_c_s.index)
        t_idx = t_idx[t_idx >= start_10y]
        if len(t_idx) > 250:
            yrs = (t_idx[-1] - t_idx[0]).days / 365.25
            tr_p_cagr = float((tr_p_s.loc[t_idx[-1]] / tr_p_s.loc[t_idx[0]]) ** (1.0 / yrs) - 1.0)
            tr_c_cagr = float((tr_c_s.loc[t_idx[-1]] / tr_c_s.loc[t_idx[0]]) ** (1.0 / yrs) - 1.0)
            tr_diff = tr_p_cagr - tr_c_cagr

    # 커스텀 재검토선 체크
    cust = PREF_CONFIG["CUSTOM_THRESHOLDS"].get(p_tk)
    cust_alert = ""
    if cust and cur_ratio > cust["ratio_high"]:
        cust_alert = f"기준선({cust['ratio_high']:.3f}) 돌파!"
    elif cust:
        cust_alert = f"기준선({cust['ratio_high']:.3f}) 하회(유지)"

    # 매수 판독
    if liq_p_eok < PREF_CONFIG["MIN_LIQ_EOK"]:
        verdict = "유동성 주의"
    elif pct_pos <= 25.0:
        verdict = "우선주 매우 유리"
    elif pct_pos <= 50.0:
        verdict = "우선주 유리"
    elif pct_pos <= 70.0:
        verdict = "중립(중앙값 부근)"
    else:
        verdict = "보통주 검토(할인축소)"

    is_held = p_tk in HELD_PREF_SET
    PAIR_RESULTS.append({
        "is_held": is_held,
        "구분": "보유중" if is_held else "관찰",
        "우선주": f"{p_nm}({p_tk})",
        "보통주": c_nm,
        "p_tk": p_tk, "p_nm": p_nm, "c_nm": c_nm,
        "cur_ratio": cur_ratio, "med_ratio": med_ratio, "q25_ratio": q25_ratio, "q75_ratio": q75_ratio,
        "min_ratio": float(ratio.min()), "max_ratio": float(ratio.max()),
        "pct_pos": pct_pos, "cur_disc": cur_disc,
        "liq_p_eok": liq_p_eok,
        "yld_p": yld_p, "yld_c": yld_c, "yld_spread": yld_spread,
        "tr_p_cagr": tr_p_cagr, "tr_c_cagr": tr_c_cagr, "tr_diff": tr_diff,
        "cust_alert": cust_alert,
        "판정": verdict,
    })

# 보유중 최상단 -> 10년 백분위 낮은 순(우선주가 역사적으로 싼 순) 정렬
PAIR_RESULTS.sort(key=lambda x: (not x["is_held"], x["pct_pos"]))

SUMMARY_DF = pd.DataFrame([
    {
        "구분": r["구분"],
        "우선주": r["우선주"],
        "현재 비율(우/보)": f"{r['cur_ratio']:.3f}",
        "10년 중앙값": f"{r['med_ratio']:.3f}",
        "현재 할인율": f"{r['cur_disc']:.1%}",
        "10년 백분위": f"{r['pct_pos']:.0f}%",
        "우선주 배당률": "-" if r["yld_p"] is None else f"{r['yld_p']:.2%}",
        "배당격차(우-보)": "-" if r["yld_spread"] is None else f"{r['yld_spread']*100:+.2f}%p",
        "10Y TR(우 vs 보)": "-" if r["tr_p_cagr"] is None else f"{r['tr_p_cagr']:+.1%} vs {r['tr_c_cagr']:+.1%}",
        "60일 거래대금": f"{r['liq_p_eok']:.1f}억",
        "판정": r["판정"],
        "비고": r["cust_alert"],
    }
    for r in PAIR_RESULTS
])

print(f"■ [우선주 vs 보통주 10년 괴리율 · 총수익률 종합 판독 ({as_of_str})]")
display(SUMMARY_DF)

In [ ]:
# ==============================================================================
# 5. 고가독성 시각화 차트 3종 + 초보자도 바로 읽히는 A4 PDF/HTML 리포트 생성 & Drive 저장
# ==============================================================================
def export_html_to_pdf(html_str: str, pdf_path: str, chart_b64_list: list[str]) -> bool:
    try:
        from weasyprint import HTML as WP_HTML
        WP_HTML(string=html_str).write_pdf(pdf_path)
        return True
    except Exception as e:
        print(f"[안내] WeasyPrint 폴백({type(e).__name__}: {e}) -> Matplotlib PDF로 생성")
    try:
        from matplotlib.backends.backend_pdf import PdfPages
        import matplotlib.image as mpimg
        with PdfPages(pdf_path) as pdf:
            for b64 in chart_b64_list:
                img = mpimg.imread(io.BytesIO(base64.b64decode(b64)), format="png")
                fig_p = plt.figure(figsize=(11.69, 8.27))
                plt.imshow(img); plt.axis("off"); plt.tight_layout(); pdf.savefig(fig_p, dpi=150); plt.close(fig_p)
        return True
    except Exception:
        return False

ys = np.arange(len(PAIR_RESULTS))
labels = [f"{'★ ' if r['is_held'] else ''}{r['p_nm']}\n({r['c_nm']})" for r in PAIR_RESULTS]

# ------------------------------------------------------------------------------
# 차트 ① (단독 풀폭): 지난 10년 중 오늘 우선주가 얼마나 저렴한가?
# ------------------------------------------------------------------------------
fig1, ax1 = plt.subplots(figsize=(13.8, 6.2), facecolor="white")
for i, r in enumerate(PAIR_RESULTS):
    if r["is_held"]:
        ax1.axhspan(i - 0.44, i + 0.44, color="#ECFDF5", alpha=0.85, zorder=0)
    ax1.plot([r["min_ratio"], r["max_ratio"]], [i, i], color="#CBD5E1", lw=3.2, solid_capstyle="round", zorder=1,
             label="10년 최저~최고 전체 범위" if i == 0 else "")
    ax1.plot([r["q25_ratio"], r["q75_ratio"]], [i, i], color="#64748B", lw=9.5, alpha=0.55, solid_capstyle="butt", zorder=2,
             label="10년 평소 구간 (하위 25% ~ 상위 75%)" if i == 0 else "")
    ax1.scatter(r["med_ratio"], i, color="#0F172A", marker="D", s=46, zorder=3,
                label="10년 중앙값 (평균 기준점)" if i == 0 else "")
    pt_col = "#059669" if r["pct_pos"] <= 35 else ("#DC2626" if r["pct_pos"] >= 70 else "#D97706")
    ax1.scatter(r["cur_ratio"], i, color=pt_col, s=150, edgecolors="#0F172A", lw=1.2, zorder=4,
                label="오늘 가격비율 (녹색=저평가 · 주황=보통 · 적색=할인축소)" if i == 0 else "")
    # 우측 고정 정렬 텍스트 컬럼 (바와 절대 겹치지 않음)
    tag_txt = f"오늘 {r['cur_ratio']:.3f}  |  할인 {r['cur_disc']:.1%}  |  10년 중 하위 {r['pct_pos']:.0f}%"
    ax1.text(1.03, i, tag_txt, va="center", ha="left", fontsize=9.4, fontweight="bold", color=pt_col)

ax1.axvline(1.0, color="#94A3B8", ls=":", lw=1.0)
ax1.set_yticks(ys)
ax1.set_yticklabels(labels, fontsize=9.6, fontweight="bold", color="#0F172A")
ax1.invert_yaxis()
ax1.set_xlim(0.24, 1.44)
ax1.set_xticks([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
ax1.set_xticklabels(["0.30\n(70% 할인)", "0.40\n(60% 할인)", "0.50\n(50% 할인)", "0.60\n(40% 할인)",
                     "0.70\n(30% 할인)", "0.80\n(20% 할인)", "0.90\n(10% 할인)", "1.00\n(동일가격)"], fontsize=8.8)
ax1.set_xlabel("가격비율 = 우선주 주가 ÷ 보통주 주가   (왼쪽으로 갈수록 보통주 대비 우선주가 저렴함)", fontsize=10.2, fontweight="bold", labelpad=8)
ax1.set_title(f"① 지난 10년 대비 오늘 우선주 가격 위치 ({as_of_str}) — ★ 연녹색 음영 = 보유 우선주",
              fontsize=13.2, fontweight="bold", color="#0F2922", loc="left", pad=28)
ax1.legend(loc="upper center", bbox_to_anchor=(0.48, 1.08), ncol=2, fontsize=8.8, frameon=True, facecolor="#F8FAFC", edgecolor="#CBD5E1")
ax1.grid(True, axis="x", color="#E2E8F0", ls="--", lw=0.8)
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
fig1.tight_layout()
buf1 = io.BytesIO(); fig1.savefig(buf1, format="png", dpi=170, bbox_inches="tight", facecolor="white")
CHART_DISC_B64 = base64.b64encode(buf1.getvalue()).decode()
if IN_COLAB: plt.show()
plt.close(fig1)

# ------------------------------------------------------------------------------
# 차트 ② (단독 풀폭): 지난 10년 실제 총수익률(배당재투자 CAGR) 우선주 vs 보통주 나란히 비교
# ------------------------------------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(13.8, 6.2), facecolor="white")
p_trs = [(r["tr_p_cagr"] * 100 if r["tr_p_cagr"] is not None else 0.0) for r in PAIR_RESULTS]
c_trs = [(r["tr_c_cagr"] * 100 if r["tr_c_cagr"] is not None else 0.0) for r in PAIR_RESULTS]
h = 0.35

for i, r in enumerate(PAIR_RESULTS):
    if r["is_held"]:
        ax2.axhspan(i - 0.45, i + 0.45, color="#ECFDF5", alpha=0.85, zorder=0)

bars_p = ax2.barh(ys - h / 2, p_trs, height=h, color="#0F766E", label="우선주 10년 연평균 총수익률 (배당 재투자 포함)", zorder=3)
bars_c = ax2.barh(ys + h / 2, c_trs, height=h, color="#94A3B8", label="보통주 10년 연평균 총수익률 (배당 재투자 포함)", zorder=3)
ax2.axvline(0, color="#334155", lw=1.2, zorder=4)

min_x = min(min(p_trs), min(c_trs), -5.0)
max_x = max(max(p_trs), max(c_trs), 15.0)
ax2.set_xlim(min_x - 4.5, max_x + 12.5)

for i, (vp, vc, r) in enumerate(zip(p_trs, c_trs, PAIR_RESULTS)):
    if r["tr_p_cagr"] is None:
        continue
    ax2.text(vp + (0.45 if vp >= 0 else -0.45), i - h / 2, f"{vp:+.1f}%", va="center",
             ha="left" if vp >= 0 else "right", fontsize=8.6, fontweight="bold", color="#0F766E")
    ax2.text(vc + (0.45 if vc >= 0 else -0.45), i + h / 2, f"{vc:+.1f}%", va="center",
             ha="left" if vc >= 0 else "right", fontsize=8.4, color="#475569")
    diff = vp - vc
    win_col = "#059669" if diff >= 0 else "#DC2626"
    win_txt = f"우선주 {diff:+.1f}%p 승" if diff >= 0 else f"보통주 {abs(diff):.1f}%p 승"
    ax2.text(max_x + 3.8, i, win_txt, va="center", ha="left", fontsize=9.1, fontweight="bold", color=win_col,
             bbox=dict(boxstyle="round,pad=0.25", facecolor="#F0FDF4" if diff >= 0 else "#FEF2F2", edgecolor=win_col, lw=0.7))

ax2.set_yticks(ys)
ax2.set_yticklabels(labels, fontsize=9.6, fontweight="bold", color="#0F172A")
ax2.invert_yaxis()
ax2.set_xlabel("최근 10년 연평균 총수익률 CAGR (%, 배당 재투자 기준 · 0% 왼쪽은 10년간 손실 구간)", fontsize=10.2, fontweight="bold", labelpad=8)
ax2.set_title("② 지난 10년 배당 포함 실제 총수익률 대결: 우선주 vs 보통주 (Yahoo Finance 10Y TR)",
              fontsize=13.2, fontweight="bold", color="#0F2922", loc="left", pad=28)
ax2.legend(loc="upper center", bbox_to_anchor=(0.45, 1.08), ncol=2, fontsize=9.2, frameon=True, facecolor="#F8FAFC", edgecolor="#CBD5E1")
ax2.grid(True, axis="x", color="#E2E8F0", ls="--", lw=0.8)
ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
fig2.tight_layout()
buf2 = io.BytesIO(); fig2.savefig(buf2, format="png", dpi=170, bbox_inches="tight", facecolor="white")
CHART_TR_B64 = base64.b64encode(buf2.getvalue()).decode()
if IN_COLAB: plt.show()
plt.close(fig2)

# ------------------------------------------------------------------------------
# 차트 ③ (단독 풀폭 2x2): 보유 우선주 3종 + 핵심 1종 10년 가격비율 시계열 (매수/주의 존 컬러 음영)
# ------------------------------------------------------------------------------
focus_list = PAIR_RESULTS[:4]
fig3, axes = plt.subplots(2, 2, figsize=(14.2, 7.8), facecolor="white")
fig3.suptitle("③ 핵심 우선주 4종 최근 10년 가격비율(우선주 ÷ 보통주) 흐름과 저평가·과열 구간",
              fontsize=13.8, fontweight="bold", color="#0F2922", y=0.985)

for idx_ax, (ax, r) in enumerate(zip(axes.flat, focus_list)):
    s = RATIO_SERIES[r["p_tk"]]
    y_lo = max(0.15, float(s.min()) - 0.04)
    y_hi = min(1.08, float(s.max()) + 0.05)
    # 구간 음영: 하위 25% 녹색(우선주 저평가), 중간 25~75% 연회색, 상위 25% 연적색(할인축소)
    ax.axhspan(y_lo, r["q25_ratio"], color="#DCFCE7", alpha=0.55, zorder=0, label="우선주 저평가 구간 (하위 25%)" if idx_ax == 0 else "")
    ax.axhspan(r["q25_ratio"], r["q75_ratio"], color="#F1F5F9", alpha=0.70, zorder=0, label="10년 평소 구간 (25~75%)" if idx_ax == 0 else "")
    ax.axhspan(r["q75_ratio"], y_hi, color="#FEE2E2", alpha=0.45, zorder=0, label="할인 축소 구간 (상위 25%)" if idx_ax == 0 else "")

    ax.plot(s.index, s, color="#0F766E", lw=1.6, zorder=2, label="일별 가격비율 (우÷보)" if idx_ax == 0 else "")
    ax.axhline(r["med_ratio"], color="#0F172A", ls="--", lw=1.2, zorder=3, label="10년 중앙값" if idx_ax == 0 else "")
    cust = PREF_CONFIG["CUSTOM_THRESHOLDS"].get(r["p_tk"])
    if cust:
        ax.axhline(cust["ratio_high"], color="#DC2626", ls=":", lw=1.8, zorder=4)
        ax.text(s.index[0], cust["ratio_high"] + 0.012, f"재검토선 {cust['ratio_high']:.3f}",
                fontsize=8.2, fontweight="bold", color="#DC2626", va="bottom",
                bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="#DC2626", alpha=0.92, lw=0.7), zorder=6)

    # 최신 시점 강조 점 + 우측 여백 말풍선
    last_dt = s.index[-1]
    pt_col = "#059669" if r["pct_pos"] <= 35 else ("#DC2626" if r["pct_pos"] >= 70 else "#D97706")
    ax.scatter([last_dt], [r["cur_ratio"]], color=pt_col, s=75, edgecolors="#0F172A", lw=1.0, zorder=5)
    ax.annotate(
        f"오늘 {r['cur_ratio']:.3f}\n(할인 {r['cur_disc']:.1%})",
        xy=(last_dt, r["cur_ratio"]),
        xytext=(-12, 16 if r["cur_ratio"] < r["med_ratio"] else -26),
        textcoords="offset points",
        ha="right",
        fontsize=8.4,
        fontweight="bold",
        color=pt_col,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor=pt_col, lw=1.0),
        arrowprops=dict(arrowstyle="->", color=pt_col, lw=1.0),
    )

    ax.set_ylim(y_lo, y_hi)
    role_prefix = "★ [보유중] " if r["is_held"] else "[관찰] "
    ax.set_title(f"{role_prefix}{r['p_nm']} ÷ {r['c_nm']}  |  10년 하위 {r['pct_pos']:.0f}% ({r['판정']})",
                 fontsize=10.6, fontweight="bold", color="#0F2922", loc="left", pad=6)
    ax.set_ylabel("가격비율 (우÷보)", fontsize=8.8)
    ax.grid(True, color="#CBD5E1", ls=":", lw=0.7)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

handles, leg_labels = axes.flat[0].get_legend_handles_labels()
fig3.legend(handles, leg_labels, loc="upper center", bbox_to_anchor=(0.5, 0.945), ncol=5, fontsize=8.6,
            frameon=True, facecolor="#F8FAFC", edgecolor="#CBD5E1")
fig3.tight_layout(rect=[0, 0, 1, 0.91])
buf3 = io.BytesIO(); fig3.savefig(buf3, format="png", dpi=170, bbox_inches="tight", facecolor="white")
CHART_TS_B64 = base64.b64encode(buf3.getvalue()).decode()
if IN_COLAB: plt.show()
plt.close(fig3)

# ------------------------------------------------------------------------------
# HTML / PDF 리포트 조립 (보유주 스포트라이트 + 30초 읽는 법 + 색상 배지 비교표 + 풀폭 차트 3장)
# ------------------------------------------------------------------------------
def verdict_pill(v: str) -> str:
    if v in ("우선주 매우 유리", "우선주 유리"):
        cls = "p-green"
    elif v in ("중립(중앙값 부근)", "유동성 주의"):
        cls = "p-amber"
    else:
        cls = "p-red"
    return f'<span class="pill {cls}">{escape(v)}</span>'

held_list = [r for r in PAIR_RESULTS if r["is_held"]]
watch_list = [r for r in PAIR_RESULTS if not r["is_held"]]
held_fav_n = sum(1 for r in held_list if "우선주" in r["판정"])
held_tr_win_n = sum(1 for r in held_list if (r["tr_diff"] or 0) >= 0)
held_avg_disc = (sum(r["cur_disc"] for r in held_list) / len(held_list)) if held_list else (sum(r["cur_disc"] for r in PAIR_RESULTS) / len(PAIR_RESULTS))

# 보유 우선주 스포트라이트 카드 생성
spotlight_items = held_list if held_list else PAIR_RESULTS[:3]
spotlight_cards = []
for r in spotlight_items:
    tr_str = "-" if r["tr_p_cagr"] is None else f"우선주 <b>{r['tr_p_cagr']:+.1%}</b> vs 보통주 {r['tr_c_cagr']:+.1%} (우선주 {r['tr_diff']*100:+.1f}%p/년)"
    div_str = "-" if r["yld_p"] is None else f"<b>{r['yld_p']:.2%}</b> (보통주보다 <b>{r['yld_spread']*100:+.2f}%p</b> 더 받음)"
    if "우선주" in r["판정"]:
        action = f"현재 보통주 대비 <b>{r['cur_disc']:.1%} 싸고</b> 10년 중 하위 {r['pct_pos']:.0f}% 구간입니다. <b>우선주 보유 유지 · 신규 적립 시 우선주 유리</b>."
    elif "보통주" in r["판정"]:
        action = f"할인이 크게 줄어 10년 상위 {100-r['pct_pos']:.0f}% 구간입니다. <b>신규 매수는 보통주 우선 검토</b> 권장."
    else:
        action = f"10년 중앙값({r['med_ratio']:.3f}) 부근의 중립 구간입니다. 기존 보유 유지."
    if r["cust_alert"]:
        action += f" <span class='alert-chip'>{escape(r['cust_alert'])}</span>"
    spotlight_cards.append(f"""
    <div class="spot-card">
      <div class="spot-head">
        <div><span class="star-tag">{'★ 보유중' if r['is_held'] else '핵심'}</span> <b>{escape(r['p_nm'])}</b> <span class="muted">({escape(r['p_tk'])} · 본주 {escape(r['c_nm'])})</span></div>
        <div>{verdict_pill(r['판정'])}</div>
      </div>
      <div class="spot-metrics">
        <div><span class="m-lbl">보통주 대비 할인율</span><div class="m-val">{r['cur_disc']:.1%} <span class="m-sub">(비율 {r['cur_ratio']:.3f})</span></div></div>
        <div><span class="m-lbl">10년 가격 위치</span><div class="m-val">하위 {r['pct_pos']:.0f}% <span class="m-sub">(중앙값 {r['med_ratio']:.3f})</span></div></div>
        <div><span class="m-lbl">우선주 배당 보너스</span><div class="m-val-sm">{div_str}</div></div>
        <div><span class="m-lbl">10년 총수익(배당포함)</span><div class="m-val-sm">{tr_str}</div></div>
      </div>
      <div class="spot-action">👉 {action}</div>
    </div>
    """)

# 깔끔한 비교표 (핵심 10컬럼 + 색상 배지 + 보유 행 강조)
tbl_rows = []
for r in PAIR_RESULTS:
    row_cls = "row-held" if r["is_held"] else ""
    tr_cell = "-" if r["tr_p_cagr"] is None else f"<b>{r['tr_p_cagr']:+.1%}</b> vs {r['tr_c_cagr']:+.1%}"
    tr_win = "-" if r["tr_diff"] is None else (f"<span class='txt-green'>우 {r['tr_diff']*100:+.1f}%p</span>" if r["tr_diff"] >= 0 else f"<span class='txt-red'>보 {abs(r['tr_diff'])*100:.1f}%p</span>")
    yld_cell = "-" if r["yld_p"] is None else f"{r['yld_p']:.2%} <span class='muted'>({r['yld_spread']*100:+.2f}%p)</span>"
    pct_col = "txt-green" if r["pct_pos"] <= 35 else ("txt-red" if r["pct_pos"] >= 70 else "txt-amber")
    tbl_rows.append(
        f"<tr class='{row_cls}'>"
        f"<td class='col-left'>{'★ 보유' if r['is_held'] else '관찰'}</td>"
        f"<td class='col-left'><b>{escape(r['p_nm'])}</b> <span class='muted'>({escape(r['c_nm'])})</span></td>"
        f"<td><b>{r['cur_disc']:.1%}</b> <span class='muted'>({r['cur_ratio']:.3f})</span></td>"
        f"<td>{1.0 - r['med_ratio']:.1%} <span class='muted'>({r['med_ratio']:.3f})</span></td>"
        f"<td class='{pct_col}'><b>하위 {r['pct_pos']:.0f}%</b></td>"
        f"<td>{yld_cell}</td>"
        f"<td>{tr_cell}</td>"
        f"<td>{tr_win}</td>"
        f"<td>{r['liq_p_eok']:.1f}억</td>"
        f"<td>{verdict_pill(r['판정'])}</td>"
        f"</tr>"
    )

# 관찰 종목 주의사항 자동 생성 (할인율은 큰데 10년 TR이 마이너스인 함정 + 할인축소 종목)
trap_names = [r["p_nm"] for r in watch_list if r["pct_pos"] <= 20 and (r["tr_p_cagr"] or 0) < 0]
exp_names = [r["p_nm"] for r in watch_list if r["pct_pos"] >= 70]
low_liq_names = [r["p_nm"] for r in watch_list if r["liq_p_eok"] < PREF_CONFIG["MIN_LIQ_EOK"]]

watch_notes = []
if trap_names:
    watch_notes.append(f"<b>① 싸 보이지만 주의할 종목 ({', '.join(trap_names)}):</b> 10년 백분위 하위 10% 이내로 할인율은 매우 크지만, 본업 실적 부진으로 <b>10년 총수익률 자체가 마이너스</b>입니다. 괴리율만 보고 매수하지 말고 본주 실적 턴어라운드 확인이 먼저입니다.")
if exp_names:
    watch_notes.append(f"<b>② 할인이 많이 줄어든 종목 ({', '.join(exp_names)}):</b> 가격비율이 10년 상위 30% 이상으로 올라와 우선주의 가격 메리트가 줄어든 구간입니다. 신규 매수 시 본주(보통주)를 함께 비교하세요.")
if low_liq_names:
    watch_notes.append(f"<b>③ 거래대금 주의 ({', '.join(low_liq_names)}):</b> 60일 일평균 거래대금이 {PREF_CONFIG['MIN_LIQ_EOK']:.0f}억 원 미만이라 대량 매매 시 호가 공백(슬리피지)에 유의해야 합니다.")
watch_notes_html = "<br>".join(watch_notes) if watch_notes else "현재 관찰 종목 중 특이 함정 경보는 없습니다."

spotlight_html = "".join(spotlight_cards)
tbl_rows_html = "".join(tbl_rows)

html_report = f"""<!doctype html>
<html lang="ko"><head><meta charset="utf-8"><title>우선주 vs 보통주 10년 비교 리포트 ({as_of_str})</title>
<style>
  @page {{ size: A4 landscape; margin: 9mm 10mm; }}
  body {{ margin:0; padding:10px; background:#F8FAFC; color:#0F172A; font-family:'NanumGothic',sans-serif; font-size:10.3px; line-height:1.5; }}
  .wrap {{ max-width:1120px; margin:0 auto; }}
  .head {{ background:linear-gradient(135deg,#0F2922,#134E4A); color:#fff; padding:15px 20px; border-radius:10px; margin-bottom:10px; border-bottom:4px solid #D4A34A; }}
  .head h1 {{ margin:0 0 4px; font-size:17.5px; }}
  .head .sub {{ font-size:10px; opacity:0.92; }}
  .kpi-row {{ display:grid; grid-template-columns:repeat(3,1fr); gap:10px; margin-bottom:10px; }}
  .kpi {{ background:#fff; border:1px solid #E2E8F0; border-top:3px solid #0F766E; border-radius:8px; padding:9px 12px; }}
  .kpi .k-lbl {{ font-size:9px; color:#64748B; font-weight:700; }}
  .kpi .k-val {{ font-size:16px; font-weight:800; color:#0F2922; margin:2px 0; }}
  .kpi .k-sub {{ font-size:8.6px; color:#475569; }}
  .guide-box {{ background:#F0FDF9; border:1px solid #99F6E4; border-left:4px solid #0F766E; border-radius:8px; padding:9px 13px; margin-bottom:10px; font-size:9.4px; }}
  .guide-grid {{ display:grid; grid-template-columns:repeat(3,1fr); gap:10px; margin-top:5px; }}
  .card {{ background:#fff; border:1px solid #E2E8F0; border-radius:10px; padding:11px 14px; margin-bottom:10px; break-inside:avoid; }}
  .card h2 {{ margin:0 0 8px; font-size:12.8px; color:#0F2922; border-bottom:1.5px solid #E2E8F0; padding-bottom:5px; }}
  .spot-grid {{ display:grid; grid-template-columns:repeat(3,1fr); gap:10px; }}
  .spot-card {{ background:#FAFDFC; border:1.2px solid #A7F3D0; border-radius:9px; padding:10px 11px; }}
  .spot-head {{ display:flex; justify-content:space-between; align-items:center; margin-bottom:7px; font-size:11px; }}
  .star-tag {{ background:#0F2922; color:#FDE68A; padding:1.5px 6px; border-radius:4px; font-size:8.3px; font-weight:700; margin-right:3px; }}
  .spot-metrics {{ display:grid; grid-template-columns:1fr 1fr; gap:6px; margin-bottom:7px; }}
  .spot-metrics > div {{ background:#fff; border:1px solid #E2E8F0; border-radius:6px; padding:5px 7px; }}
  .m-lbl {{ font-size:8px; color:#64748B; display:block; }}
  .m-val {{ font-size:12px; font-weight:800; color:#0F2922; }}
  .m-val-sm {{ font-size:9px; color:#1E293B; margin-top:1px; }}
  .m-sub {{ font-size:8.2px; color:#64748B; font-weight:400; }}
  .spot-action {{ background:#ECFDF5; border-radius:6px; padding:6px 8px; font-size:8.9px; color:#065F46; }}
  .alert-chip {{ display:inline-block; background:#FEF3C7; color:#92400E; padding:0.5px 5px; border-radius:4px; font-size:8px; font-weight:700; margin-left:3px; }}
  table.tbl {{ width:100%; border-collapse:collapse; font-size:9.3px; }}
  table.tbl th {{ background:#0F2922; color:#fff; padding:6.5px 6px; text-align:right; font-weight:700; }}
  table.tbl td {{ padding:6.5px 6px; border-bottom:1px solid #E2E8F0; text-align:right; }}
  table.tbl tr.row-held {{ background:#ECFDF5; }}
  table.tbl td.col-left, table.tbl th:first-child, table.tbl th:nth-child(2) {{ text-align:left; }}
  .pill {{ display:inline-block; padding:2px 8px; border-radius:999px; font-size:8.4px; font-weight:700; white-space:nowrap; }}
  .p-green {{ background:#DCFCE7; color:#166534; }} .p-amber {{ background:#FEF3C7; color:#92400E; }} .p-red {{ background:#FEE2E2; color:#991B1B; }}
  .txt-green {{ color:#059669; }} .txt-amber {{ color:#D97706; }} .txt-red {{ color:#DC2626; }} .muted {{ color:#64748B; font-size:8.4px; }}
  .note-box {{ background:#FFFBEB; border:1px solid #FDE68A; border-left:4px solid #D97706; border-radius:7px; padding:8px 11px; margin-top:9px; font-size:9.1px; color:#78350F; line-height:1.55; }}
  .chart-tip {{ background:#F8FAFC; border:1px solid #E2E8F0; border-left:4px solid #0F766E; border-radius:6px; padding:7px 10px; margin-top:8px; font-size:9.2px; color:#334155; }}
  img.chart {{ width:100%; height:auto; display:block; }}
  .page-break {{ page-break-before:always; break-before:page; }}
</style></head><body><div class="wrap">

  <!-- PAGE 1: 한눈에 보는 결론 + 30초 가이드 + 내 보유 우선주 스포트라이트 -->
  <div class="head">
    <h1>한국 우량 우선주 vs 보통주 10년 비교 · 괴리율 판독 리포트 ({as_of_str})</h1>
    <div class="sub">네이버 3,000일 수정주가(10년 괴리율 분포) + Yahoo Finance 10년 배당재투자 총수익률(TR) 기준 · ★ 표시 = 내 보유 우선주</div>
  </div>

  <div class="kpi-row">
    <div class="kpi">
      <div class="k-lbl">내 보유 우선주 현재 진단</div>
      <div class="k-val">{held_fav_n} / {len(held_list) if held_list else len(PAIR_RESULTS)}종 「우선주 유리」</div>
      <div class="k-sub">보유 중인 우선주 모두 10년 평균보다 저렴한 구간</div>
    </div>
    <div class="kpi">
      <div class="k-lbl">보유 우선주 평균 할인율</div>
      <div class="k-val">{held_avg_disc:.1%} 할인</div>
      <div class="k-sub">같은 기업 보통주 대비 평균 {held_avg_disc:.1%} 저렴하게 거래 중</div>
    </div>
    <div class="kpi">
      <div class="k-lbl">지난 10년 배당포함 총수익 승률</div>
      <div class="k-val">보유 {held_tr_win_n} / {len(held_list) if held_list else len(PAIR_RESULTS)}종 우선주 승</div>
      <div class="k-sub">장기 배당 재투자 시 우선주가 보통주 수익률을 상회</div>
    </div>
  </div>

  <div class="guide-box">
    <b>📖 30초 만에 읽는 이 리포트 핵심 용어 가이드</b>
    <div class="guide-grid">
      <div><b>1) 할인율 · 가격비율(우÷보):</b><br>비율이 <b>0.525</b>면 우선주가 보통주 가격의 52.5%, 즉 <b>47.5% 더 싸다</b>는 뜻입니다. 비율이 낮을수록(할인율이 클수록) 우선주가 유리합니다.</div>
      <div><b>2) 10년 백분위 (하위 N%):</b><br>지난 10년 매일의 가격비율 중 오늘이 몇 번째로 싼지 나타냅니다. <b>하위 0~35% = 저평가(우선주 유리)</b>, 50% = 평소, <b>70%↑ = 할인 축소(보통주 검토)</b>.</div>
      <div><b>3) 10년 총수익률 (TR, CAGR):</b><br>단순 주가뿐 아니라 <b>매년 받은 배당을 재투자했을 때의 연평균 수익률</b>입니다. 우선주는 싸게 사서 배당률이 높기 때문에 장기 복리에서 유리할 때가 많습니다.</div>
    </div>
  </div>

  <div class="card">
    <h2>01. 내 보유 우선주 정밀 진단 카드 (유지 · 적립 · 교체 기준 한눈에 보기)</h2>
    <div class="spot-grid">{spotlight_html}</div>
  </div>

  <!-- PAGE 2: 전체 10종목 한눈에 비교표 + 관찰 종목 주의 코멘트 -->
  <div class="card page-break">
    <h2>02. 전체 {len(PAIR_RESULTS)}종목 우선주 vs 보통주 한눈에 비교표 (★ 연녹색 행 = 보유 우선주)</h2>
    <table class="tbl">
      <thead><tr>
        <th>구분</th><th>우선주 (본주)</th><th>현재 할인율 (비율)</th><th>10년 평소 할인율</th>
        <th>10년 가격 위치</th><th>우선주 배당률 (격차)</th><th>10년 총수익 (우 vs 보)</th>
        <th>10년 승자</th><th>60일 거래대금</th><th>최종 판정</th>
      </tr></thead>
      <tbody>{tbl_rows_html}</tbody>
    </table>
    <div class="note-box">
      <b>🔍 관찰 종목 읽을 때 꼭 체크할 3가지 포인트</b><br>{watch_notes_html}
    </div>
  </div>

  <!-- PAGE 3: 차트 ① 단독 풀폭 -->
  <div class="card page-break">
    <h2>03. [차트 ①] 지난 10년 대비 오늘 우선주가 얼마나 저렴한가?</h2>
    <img class="chart" src="data:image/png;base64,{CHART_DISC_B64}">
    <div class="chart-tip">
      <b>💡 그래프 읽는 법:</b> 회색 막대는 지난 10년 동안 보통주 대비 우선주 가격이 움직인 범위이고, <b>검정 마름모(◆)가 10년 평균(중앙값)</b>입니다.
      <b>동그라미(오늘 가격)가 검정 마름모보다 왼쪽(녹색)에 있을수록 평소보다 싸다</b>는 뜻이며, 오른쪽(적색)에 있으면 할인이 줄어 보통주를 검토할 구간입니다.
    </div>
  </div>

  <!-- PAGE 4: 차트 ② 단독 풀폭 -->
  <div class="card page-break">
    <h2>04. [차트 ②] 지난 10년 배당 포함 실제 총수익률 대결: 우선주 vs 보통주</h2>
    <img class="chart" src="data:image/png;base64,{CHART_TR_B64}">
    <div class="chart-tip">
      <b>💡 그래프 읽는 법:</b> 진녹색 막대가 <b>우선주</b>, 회색 막대가 <b>보통주</b>의 지난 10년 연평균 총수익률(배당 재투자 포함)입니다.
      오른쪽 녹색 박스가 뜬 종목은 <b>10년 장기 보유 시 우선주가 보통주를 실제로 이긴 종목</b>입니다. 반대로 0% 왼쪽(마이너스)으로 뻗은 종목은 할인율이 커 보여도 본업이 부진했던 종목이므로 주의해야 합니다.
    </div>
  </div>

  <!-- PAGE 5: 차트 ③ 단독 풀폭 -->
  <div class="card page-break">
    <h2>05. [차트 ③] 핵심 우선주 4종(보유주 최우선) 10년 가격비율 추이 &amp; 매수·주의 존</h2>
    <img class="chart" src="data:image/png;base64,{CHART_TS_B64}">
    <div class="chart-tip">
      <b>💡 그래프 읽는 법:</b> 선이 아래쪽 <b>연녹색 구간(하위 25%)</b>에 가까울수록 우선주가 역사적으로 저렴해 신규 적립에 유리하며, 위쪽 <b>연적색 구간(상위 25%)</b>으로 올라가면 보통주와의 가격 차가 줄어들어 교체·비중 조절을 검토하는 구간입니다. 현대차3우B의 빨간 점선(<b>0.559</b>)은 개인 재검토 기준선입니다.
    </div>
  </div>

</div></body></html>"""

r_html = f"preferred_tracker_report_{run_stamp}.html"
r_pdf = f"preferred_tracker_report_{run_stamp}.pdf"
Path(r_html).write_text(html_report, encoding="utf-8")
pdf_ok = export_html_to_pdf(html_report, r_pdf, [CHART_DISC_B64, CHART_TR_B64, CHART_TS_B64])
print(f"[리포트 완료] HTML={Path(r_html).stat().st_size/1024:.1f} KB | PDF={'성공' if pdf_ok else '건너뜀'}")

# Google Drive 자동 연결 재시도 (Cell 2에서 미연결/취소된 경우에도 Cell 5에서 다시 시도)
if IN_COLAB and PREF_CONFIG.get("USE_GOOGLE_DRIVE", True) and DRIVE_REPORT_DIR is None:
    try:
        from google.colab import drive
        if Path("/content/drive/MyDrive").exists():
            pass
        else:
            drive.mount("/content/drive", force_remount=True)
        DRIVE_ROOT = Path(PREF_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[Drive 재연결 완료] {DRIVE_REPORT_DIR}")
    except Exception as e:
        print(f"[Drive 미연결] 로컬(/content/{r_pdf if pdf_ok else r_html})에 저장됨 ({type(e).__name__}: {e})")
        print("  -> 브라우저 팝업 차단 시 좌측 📁 탭의 [Drive 마운트] 아이콘을 누르거나 아래 자동 다운로드를 이용하세요.")
        try:
            from google.colab import files
            files.download(r_pdf if pdf_ok else r_html)
        except Exception:
            pass

if DRIVE_REPORT_DIR is not None:
    try:
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        (DRIVE_REPORT_DIR / r_html).write_text(html_report, encoding="utf-8")
        (DRIVE_REPORT_DIR / "preferred_tracker_report_latest.html").write_text(html_report, encoding="utf-8")
        saved_targets = [DRIVE_REPORT_DIR / r_html]
        if pdf_ok and Path(r_pdf).exists():
            pdf_bytes = Path(r_pdf).read_bytes()
            (DRIVE_REPORT_DIR / r_pdf).write_bytes(pdf_bytes)
            (DRIVE_REPORT_DIR / "preferred_tracker_report_latest.pdf").write_bytes(pdf_bytes)
            saved_targets.insert(0, DRIVE_REPORT_DIR / r_pdf)
        csv_name = f"preferred_summary_{run_stamp}.csv"
        SUMMARY_DF.to_csv(DRIVE_REPORT_DIR / csv_name, index=False, encoding="utf-8-sig")
        SUMMARY_DF.to_csv(DRIVE_REPORT_DIR / "preferred_summary_latest.csv", index=False, encoding="utf-8-sig")
        saved_targets.append(DRIVE_REPORT_DIR / csv_name)
        print(f"[Drive 저장 검증 완료] 폴더: {DRIVE_REPORT_DIR}")
        for fp in saved_targets:
            print(f"  - {fp.name} ({fp.stat().st_size/1024:.1f} KB)")
    except Exception as e:
        print(f"[주의] Drive 저장 실패({type(e).__name__}): {e}")